In [2]:
import numpy as np
from itertools import combinations
from time import perf_counter

In [3]:
# taken from https://gist.github.com/danoneata/49a807f47656fedbb389

def fvecs_read(filename, c_contiguous=True):
    """
    reads a file in the fvecs format, returning a 2D numpy array of shape (n, d)
    """
    fv = np.fromfile(filename, dtype=np.float32)
    if fv.size == 0:
        return np.zeros((0, 0))
    dim = fv.view(np.int32)[0]
    assert dim > 0
    fv = fv.reshape(-1, 1 + dim)
    if not all(fv.view(np.int32)[:, 0] == dim):
        raise IOError("Non-uniform vector sizes in " + filename)
    fv = fv[:, 1:]
    if c_contiguous:
        fv = fv.copy()
    return fv

In [4]:
# taken from https://gist.github.com/CharlesLiu7/0496773a8a21934a746e3fd496f11e0c

def ivecs_read(fname):
    a = np.fromfile(fname, dtype='int32')
    d = a[0]
    return a.reshape(-1, d + 1)[:, 1:].copy()

In [5]:
sim_hash_dimension = 64

def sim_hash(doc, m=sim_hash_dimension, d=128, seed=42):
    np.random.seed(seed)
    R = np.random.randn(m, d).astype('float32')
    sim_hashes = np.dot(doc, R.T) > 0
    return sim_hashes.astype('uint8') #! should be bits

In [6]:
docs = fvecs_read('siftsmall/siftsmall_base.fvecs')
print(sim_hash(docs[0]))
print(sim_hash(docs[1]))

[0 1 1 1 1 0 1 1 1 0 1 1 1 1 0 1 1 1 1 0 1 1 1 0 0 1 1 1 1 0 1 0 0 0 0 1 0
 0 0 1 1 0 0 1 0 0 0 1 0 0 0 0 1 0 0 1 1 0 1 0 1 0 1 0]
[0 0 1 1 1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 0 0 1 1 0 0 1 0 1 1 0 0 1 0 0 0 1 0
 0 0 1 1 0 1 0 0 0 1 1 0 0 0 1 0 0 0 1 1 0 0 0 1 0 0 0]


In [7]:
def nn_distance(query, data, k=10):
    dist = np.count_nonzero(query != data) #! can be bitwise AND
    if dist < k:
        return dist
    else:
        return None
    # if dist < k:
    #     return dist

## Bruteforce Near-Duplicate Detection with SimHash

just a baseline to confront with spark implementation

In [8]:
start = perf_counter()
hashes = [sim_hash(d) for d in docs]
end = perf_counter()
print(f"Time to compute simhashes: {end - start}")

close_pairs = []

total_running_time = 0

for i, j in combinations(range(len(hashes)), 2):
    start = perf_counter()
    result = nn_distance(hashes[i], hashes[j], k=2)
    total_running_time += perf_counter() - start
    if result is not None:
        close_pairs.append((i, j, result))
        print(f"Distance between index {i} and {j}: {result}")

print(f"Total running time: {total_running_time}")
print(f"Average running time per pair: {total_running_time / len(close_pairs) if close_pairs else 0}")
print(f"Number of close pairs: {len(close_pairs)}")

Time to compute simhashes: 1.7007307999883778
Distance between index 8 and 20: 1
Distance between index 10 and 18: 1
Distance between index 21 and 4259: 1
Distance between index 36 and 1905: 1
Distance between index 36 and 2336: 0
Distance between index 44 and 6970: 1
Distance between index 46 and 2021: 1
Distance between index 46 and 2497: 0
Distance between index 46 and 5148: 1
Distance between index 46 and 5479: 1
Distance between index 46 and 5507: 1
Distance between index 46 and 6009: 0
Distance between index 48 and 6160: 1
Distance between index 48 and 7841: 1
Distance between index 56 and 77: 0
Distance between index 70 and 1342: 1
Distance between index 105 and 1546: 1
Distance between index 105 and 1659: 1
Distance between index 105 and 2071: 1
Distance between index 105 and 4678: 0
Distance between index 105 and 4829: 1
Distance between index 105 and 5259: 1
Distance between index 105 and 5431: 1
Distance between index 105 and 7398: 1
Distance between index 105 and 8216: 1
Di

# banded approach

## spark sanity check

In [14]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("spark-smoke-test")
    .master("local[*]")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

print("Driver Python:", sys.executable)
print("Spark version:", spark.version)

# df = spark.createDataFrame([(1, "a"), (2, "b"), (3, "c")], ["id", "letter"])
# result = df.filter(col("id") > 1).orderBy("id").collect()

# print("Result:", result)
# assert [row.id for row in result] == [2, 3]
# assert [row.letter for row in result] == ["b", "c"]

print("Spark is working correctly.")

sc = spark.sparkContext


Driver Python: c:\Python311\python.exe
Spark version: 4.1.1
Spark is working correctly.


In [21]:
def divide_bands(hash, num_bands, doc_id):
    band_size = len(hash) // num_bands
    h = [((hash[i * band_size:(i + 1) * band_size].tobytes(), i), doc_id) for i in range(num_bands)]
    return h

divide_bands(hashes[0], num_bands=4, doc_id=1)


[((b'\x00\x01\x01\x01\x01\x00\x01\x01\x01\x00\x01\x01\x01\x01\x00\x01', 0), 1),
 ((b'\x01\x01\x01\x00\x01\x01\x01\x00\x00\x01\x01\x01\x01\x00\x01\x00', 1), 1),
 ((b'\x00\x00\x00\x01\x00\x00\x00\x01\x01\x00\x00\x01\x00\x00\x00\x01', 2), 1),
 ((b'\x00\x00\x00\x00\x01\x00\x00\x01\x01\x00\x01\x00\x01\x00\x01\x00', 3), 1)]

In [ ]:
rdd = sc.parallelize(hashes)
rdd.zipWithIndex()

candidates = (
    rdd.zipWithIndex()
    .map(lambda h: divide_bands(h[0], num_bands=4, doc_id=h[1]))
    .flatMap(lambda x: x)
    .mapValues(lambda doc_id: [doc_id])
    .reduceByKey(lambda a, b: a + b)
)
# candidates.take(5)

candidate_pairs = (
    candidates
    .flatMap(lambda kv: [tuple(sorted(pair)) for pair in combinations(kv[1], 2)])  # list of (id1,id2)
    .map(lambda pair: (pair, None))
    .distinct()
    .map(lambda x: x[0])                        # (id1, id2)
)

pairs = candidate_pairs.collect()

results = []
for i, j in pairs:
    dist = nn_distance(hashes[i - 1], hashes[j - 1], k=2)
    if dist is not None:
        results.append(((i, j), dist))

In [29]:
print(results)
print(len(results))

[((5991, 6293), 1), ((7391, 7957), 0), ((4605, 5087), 1), ((479, 1029), 1), ((1632, 2024), 1), ((3379, 3777), 1), ((9825, 9843), 1), ((3458, 3902), 0), ((2022, 6010), 1), ((6458, 6678), 1), ((6507, 6657), 1), ((6480, 6688), 1), ((7454, 7922), 1), ((436, 988), 1), ((9595, 9713), 0), ((8500, 8972), 1), ((2071, 2917), 1), ((6805, 7487), 1), ((7829, 8271), 1), ((4920, 9888), 0), ((5173, 5335), 1), ((4561, 4739), 1), ((2446, 3154), 1), ((1025, 2224), 1), ((560, 8371), 1), ((1891, 2482), 1), ((7391, 8966), 1), ((9193, 9472), 1), ((5176, 5587), 1), ((4864, 4875), 1), ((8098, 9101), 1), ((4461, 4492), 0), ((4543, 4638), 0), ((113, 168), 1), ((873, 904), 1), ((464, 531), 0), ((440, 859), 1), ((2483, 3818), 1), ((6283, 8306), 1), ((2435, 3794), 1), ((9, 21), 1), ((5508, 6010), 1), ((1626, 1896), 1), ((7906, 8336), 0), ((2096, 2606), 0), ((37, 2337), 0), ((4640, 4726), 1), ((9548, 9618), 0), ((5359, 5931), 0), ((1289, 1781), 1), ((5811, 6407), 1), ((7124, 7850), 1), ((1925, 2297), 1), ((6911, 767

In [3]:
spark.stop()